# Customer Churn Analysis: SQL Part

Same Telco churn data, now answered with **SQL**. This notebook uses SQLite (built into Python, nothing to install), so the queries work the same way as in MySQL or PostgreSQL, with tiny syntax differences.

**Goal:** answer the key business questions with SQL, check the numbers match the Python analysis, and export a clean file for the Power BI dashboard.

**SQL ideas used here (each one is explained where it appears):** `SELECT`, `WHERE`, `GROUP BY`, aggregate functions, `CASE WHEN`, `HAVING`, a CTE (`WITH`), and a window function (`RANK() OVER`).

## Step 1: Load, clean, and put the data in a database
Upload `WA_Fn-UseC_-Telco-Customer-Churn.csv` to the Colab sidebar first (same file as before).
*Why:* SQL runs on tables in a database, so we load the clean DataFrame into a SQLite table called `churn`.

In [ ]:
import pandas as pd
import sqlite3

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Same cleaning as the Python notebook
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])
df['ChurnFlag'] = (df['Churn'] == 'Yes').astype(int)   # 1 = left, 0 = stayed
print('Clean rows:', len(df))

# Create the database and the table
conn = sqlite3.connect('churn.db')
df.to_sql('churn', conn, if_exists='replace', index=False)

# Small helper: run a SQL query and show the result as a table
def q(sql):
    return pd.read_sql_query(sql, conn)

q('SELECT COUNT(*) AS customers FROM churn')

## Query 1: Overall churn rate
* `COUNT(*)` counts rows, `SUM(ChurnFlag)` counts churned customers (each churner is a 1).
* `AVG(ChurnFlag)` is the share of 1s, which is the churn rate. We multiply by 100 and `ROUND` it.

*Expected to match Python:* about 26.6%.

In [ ]:
q('''
SELECT COUNT(*)                          AS customers,
       SUM(ChurnFlag)                    AS churned,
       ROUND(100.0 * AVG(ChurnFlag), 1)  AS churn_rate_pct
FROM churn
''')

## Query 2: Churn rate by contract type
* `GROUP BY Contract` makes one row per contract type, and the aggregates are calculated inside each group.
* `ORDER BY ... DESC` puts the worst group first.

*Expected:* month-to-month 42.7%, one year 11.3%, two year 2.8%.

In [ ]:
q('''
SELECT Contract,
       COUNT(*)                          AS customers,
       ROUND(100.0 * AVG(ChurnFlag), 1)  AS churn_rate_pct
FROM churn
GROUP BY Contract
ORDER BY churn_rate_pct DESC
''')

## Query 3: Churn rate by tenure band
* `CASE WHEN` creates the bands, like an if/else inside SQL.
* We group by the band we just created, and the numbers in the labels keep the order correct.

*Expected:* highest churn in the first 12 months (about 47.7%).

In [ ]:
q('''
SELECT CASE WHEN tenure <= 12 THEN '1) 0-12 months'
            WHEN tenure <= 24 THEN '2) 13-24 months'
            WHEN tenure <= 48 THEN '3) 25-48 months'
            ELSE                   '4) 49-72 months' END AS tenure_group,
       COUNT(*)                          AS customers,
       ROUND(100.0 * AVG(ChurnFlag), 1)  AS churn_rate_pct
FROM churn
GROUP BY tenure_group
ORDER BY tenure_group
''')

## Query 4: Does tech support help, inside each internet type?
* Grouping by **two columns** compares every combination.
* `WHERE InternetService != 'No'` removes customers with no internet, since they can't have tech support.

*Why it matters:* it checks that the tech support effect isn't just because of internet type.

In [ ]:
q('''
SELECT InternetService,
       TechSupport,
       COUNT(*)                          AS customers,
       ROUND(100.0 * AVG(ChurnFlag), 1)  AS churn_rate_pct
FROM churn
WHERE InternetService != 'No'
GROUP BY InternetService, TechSupport
ORDER BY churn_rate_pct DESC
''')

## Query 5: The highest-risk segment
* `WHERE` with several `AND` conditions filters to one segment: month-to-month, fiber optic, tenure 12 months or less.

*Expected:* about 70.2% churn.

In [ ]:
q('''
SELECT COUNT(*)                          AS customers,
       SUM(ChurnFlag)                    AS churned,
       ROUND(100.0 * AVG(ChurnFlag), 1)  AS churn_rate_pct
FROM churn
WHERE Contract = 'Month-to-month'
  AND InternetService = 'Fiber optic'
  AND tenure <= 12
''')

## Query 6: Where is the lost revenue coming from? (CTE)
* A **CTE** (`WITH ... AS`) is a named temporary result. It splits a hard query into two readable steps.
* Step 1 (`contract_loss`): monthly revenue lost from churned customers, per contract.
* Step 2: each contract's share of the total lost revenue.

In [ ]:
q('''
WITH contract_loss AS (
    SELECT Contract,
           SUM(CASE WHEN ChurnFlag = 1 THEN MonthlyCharges ELSE 0 END) AS lost_revenue
    FROM churn
    GROUP BY Contract
)
SELECT Contract,
       ROUND(lost_revenue, 0) AS lost_monthly_revenue,
       ROUND(100.0 * lost_revenue / (SELECT SUM(lost_revenue) FROM contract_loss), 1) AS pct_of_total_lost
FROM contract_loss
ORDER BY lost_revenue DESC
''')

## Query 7: Rank payment methods by churn risk (window function)
* `RANK() OVER (ORDER BY ...)` is a **window function**: it adds a rank column without collapsing the rows.
* Window functions are very common in analyst interviews.

In [ ]:
q('''
SELECT PaymentMethod,
       COUNT(*)                          AS customers,
       ROUND(100.0 * AVG(ChurnFlag), 1)  AS churn_rate_pct,
       RANK() OVER (ORDER BY AVG(ChurnFlag) DESC) AS risk_rank
FROM churn
GROUP BY PaymentMethod
''')

## Query 8: Top 5 riskiest customer groups
* We combine contract and internet type, keep only groups with at least 100 customers using `HAVING`, and show the top 5.
* `WHERE` filters rows *before* grouping, and `HAVING` filters groups *after* grouping. This is a classic interview question.

In [ ]:
q('''
SELECT Contract,
       InternetService,
       COUNT(*)                          AS customers,
       ROUND(100.0 * AVG(ChurnFlag), 1)  AS churn_rate_pct
FROM churn
GROUP BY Contract, InternetService
HAVING COUNT(*) >= 100
ORDER BY churn_rate_pct DESC
LIMIT 5
''')

## Step 2: Export a clean file for the Power BI dashboard
We add the tenure band column and save the cleaned data as `telco_churn_clean.csv`. This is the file you will load into Power BI.

In [ ]:
df['TenureGroup'] = pd.cut(df['tenure'], bins=[-1, 12, 24, 48, 72],
                           labels=['0-12 months', '13-24 months', '25-48 months', '49-72 months'])
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
df.to_csv('telco_churn_clean.csv', index=False)
print('Saved telco_churn_clean.csv with', len(df), 'rows')

# In Colab, this downloads the file to your laptop
try:
    from google.colab import files
    files.download('telco_churn_clean.csv')
except ImportError:
    pass

## Check your work
The SQL results should match the Python notebook (for example overall churn 26.6%, month-to-month 42.7%, highest-risk segment 70.2%). If a number differs, find out why before you put it on your resume.

**What SQL showed that Python also showed:** contract type, tenure, and service type drive churn. **What SQL added:** the share of lost revenue by contract (Query 6) and a ranking of payment methods and customer groups (Queries 7 and 8).